# Notebook 01 — Exploratory Data Analysis
**Project:** E-Commerce Customer Analytics Pipeline  
**Author:** Hari Etta  
**Data Source:** Fake Store API (https://fakestoreapi.com)

Findings here directly inform the Silver cleaning layer, dbt schema tests, A/B test baseline, and cohort windows.

In [ ]:
# Cell 1 — Setup
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
os.makedirs('../data/sample', exist_ok=True)

BASE_URL = 'https://fakestoreapi.com'
print(f'Setup complete. API base: {BASE_URL}')

In [ ]:
# Cell 2 — Fetch API data
def fetch_endpoint(endpoint):
    url = f'{BASE_URL}/{endpoint}'
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    print(f'  GET /{endpoint} -> {len(data)} records | status {resp.status_code}')
    return data

print('Fetching all API endpoints...')
raw_products   = fetch_endpoint('products')
raw_users      = fetch_endpoint('users')
raw_carts      = fetch_endpoint('carts')
raw_categories = fetch_endpoint('products/categories')

df_products = pd.DataFrame(raw_products)
df_users    = pd.DataFrame(raw_users)
df_carts    = pd.DataFrame(raw_carts)

print(f'\nproducts shape: {df_products.shape}')
print(f'users shape:    {df_users.shape}')
print(f'carts shape:    {df_carts.shape}')
print(f'categories:     {raw_categories}')

In [ ]:
# Cell 3 — Schema inspection (fixed: stringify dict/list columns before nunique)
def schema_report(df, name):
    df_safe = df.copy()
    for col in df_safe.columns:
        if df_safe[col].apply(lambda x: isinstance(x, (dict, list))).any():
            df_safe[col] = df_safe[col].apply(str)
    report = pd.DataFrame({
        'column':       df_safe.columns,
        'dtype':        df_safe.dtypes.values,
        'null_count':   df_safe.isnull().sum().values,
        'null_pct':     (df_safe.isnull().mean() * 100).round(2).values,
        'unique_count': df_safe.nunique().values,
        'sample_value': [
            df_safe[c].dropna().iloc[0] if df_safe[c].dropna().shape[0] > 0 else None
            for c in df_safe.columns
        ]
    })
    print(f'\n=== Schema: {name} ===')
    return report

display(schema_report(df_products, 'Products'))
display(schema_report(df_users, 'Users'))
display(schema_report(df_carts, 'Carts'))

In [ ]:
# Cell 4 — Data quality checks
print('=== DATA QUALITY CHECKS ===')

print(f'Product ID duplicates: {df_products["id"].duplicated().sum()}')
print(f'User ID duplicates:    {df_users["id"].duplicated().sum()}')
print(f'Cart ID duplicates:    {df_carts["id"].duplicated().sum()}')

valid_user_ids = set(df_users['id'])
cart_user_ids  = set(df_carts['userId'])
orphan_carts   = cart_user_ids - valid_user_ids
print(f'\nOrphan cart userIds: {orphan_carts if orphan_carts else "None — RI intact"}')

print(f'Products with price <= 0:  {(df_products["price"] <= 0).sum()}')
print(f'Products with price > 500: {(df_products["price"] > 500).sum()}')

df_carts['date'] = pd.to_datetime(df_carts['date'])
cart_items = []
for _, cart in df_carts.iterrows():
    for item in cart['products']:
        cart_items.append({
            'cart_id':    cart['id'],
            'user_id':    cart['userId'],
            'date':       cart['date'],
            'product_id': item['productId'],
            'quantity':   item['quantity']
        })
df_cart_items = pd.DataFrame(cart_items)
print(f'\nExploded cart line items: {len(df_cart_items)}')
print(f'Quantity <= 0: {(df_cart_items["quantity"] <= 0).sum()}')
display(df_cart_items.head())

In [ ]:
# Cell 5 — Product distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Product Distributions', fontsize=14, fontweight='bold')

axes[0].hist(df_products['price'], bins=15, color='steelblue', edgecolor='white')
axes[0].axvline(df_products['price'].median(), color='red', linestyle='--',
                label=f'Median: ${df_products["price"].median():.2f}')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].legend()

ratings = df_products['rating'].apply(lambda x: x['rate'] if isinstance(x, dict) else np.nan)
axes[1].hist(ratings.dropna(), bins=10, color='seagreen', edgecolor='white')
axes[1].set_title('Rating Distribution')
axes[1].set_xlabel('Rating (out of 5)')

cat_counts = df_products['category'].value_counts()
axes[2].barh(cat_counts.index, cat_counts.values, color='coral')
axes[2].set_title('Products by Category')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('../data/sample/product_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print(df_products['price'].describe().round(2))

In [ ]:
# Cell 6 — Order value analysis
df_order_lines = df_cart_items.merge(
    df_products[['id', 'price', 'category']],
    left_on='product_id', right_on='id', how='left'
)
df_order_lines['line_total'] = df_order_lines['quantity'] * df_order_lines['price']

df_orders_agg = df_order_lines.groupby('cart_id').agg(
    user_id         = ('user_id', 'first'),
    order_date      = ('date', 'first'),
    item_count      = ('product_id', 'count'),
    order_value     = ('line_total', 'sum'),
    unique_products = ('product_id', 'nunique')
).reset_index()

print('=== Order-Level Summary ===')
print(df_orders_agg.describe().round(2))
display(df_orders_agg)

In [ ]:
# Cell 7 — Customer behavior
df_customer = df_orders_agg.groupby('user_id').agg(
    order_count     = ('cart_id', 'count'),
    total_spend     = ('order_value', 'sum'),
    avg_order_value = ('order_value', 'mean'),
    first_order     = ('order_date', 'min'),
    last_order      = ('order_date', 'max')
).reset_index()
df_customer['days_active'] = (df_customer['last_order'] - df_customer['first_order']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Behavior', fontsize=14, fontweight='bold')

axes[0].barh([f'User {u}' for u in df_customer['user_id']],
             df_customer['total_spend'], color='steelblue')
axes[0].set_title('Total Spend by Customer')
axes[0].set_xlabel('Total Spend ($)')

axes[1].scatter(df_customer['order_count'], df_customer['total_spend'],
                color='coral', s=100, edgecolors='white')
axes[1].set_title('Spend vs Order Count (LTV Proxy)')
axes[1].set_xlabel('Order Count')
axes[1].set_ylabel('Total Spend ($)')

plt.tight_layout()
plt.savefig('../data/sample/customer_behavior.png', dpi=120, bbox_inches='tight')
plt.show()
print(df_customer.describe().round(2))

In [ ]:
# Cell 8 — Category revenue
category_revenue = df_order_lines.groupby('category').agg(
    total_revenue = ('line_total', 'sum'),
    total_units   = ('quantity', 'sum'),
    order_count   = ('cart_id', 'nunique')
).reset_index().sort_values('total_revenue', ascending=False)
category_revenue['revenue_share'] = (
    category_revenue['total_revenue'] / category_revenue['total_revenue'].sum() * 100
).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Category Performance', fontsize=14, fontweight='bold')
colors = ['steelblue', 'seagreen', 'coral', 'mediumpurple']

axes[0].barh(category_revenue['category'], category_revenue['total_revenue'], color=colors)
axes[0].set_title('Revenue by Category')
axes[0].set_xlabel('Revenue ($)')

axes[1].pie(category_revenue['revenue_share'], labels=category_revenue['category'],
            autopct='%1.1f%%', colors=colors)
axes[1].set_title('Revenue Share')

plt.tight_layout()
plt.savefig('../data/sample/category_revenue.png', dpi=120, bbox_inches='tight')
plt.show()
display(category_revenue)

In [ ]:
# Cell 9 — Synthetic event simulation
# The Fake Store API has no clickstream data.
# We simulate page_view -> product_view -> add_to_cart -> checkout_start -> purchase
# with realistic drop-off rates and random A/B variant assignment.
np.random.seed(42)

user_ids    = df_users['id'].tolist()
product_ids = df_products['id'].tolist()
devices     = ['mobile', 'desktop']
channels    = ['organic', 'paid_search', 'email', 'social', 'direct']
event_types = ['page_view', 'product_view', 'add_to_cart', 'checkout_start', 'purchase']

events     = []
session_id = 1000
start_date = datetime(2024, 1, 1)

for user_id in user_ids:
    n_sessions = np.random.poisson(lam=15)
    for _ in range(max(1, n_sessions)):
        session_id   += 1
        device        = np.random.choice(devices, p=[0.60, 0.40])
        channel       = np.random.choice(channels, p=[0.30, 0.25, 0.20, 0.15, 0.10])
        variant       = np.random.choice(['A', 'B'])
        purchase_rate = 0.41 if variant == 'B' else 0.38
        t = start_date + timedelta(
            days=np.random.randint(0, 90),
            hours=np.random.randint(0, 24),
            minutes=np.random.randint(0, 60)
        )

        def add_event(etype, ts, pid=None):
            events.append({
                'event_id':   f'evt_{session_id}_{etype}',
                'session_id': session_id,
                'user_id':    user_id,
                'event_type': etype,
                'device':     device,
                'channel':    channel,
                'variant':    variant,
                'product_id': pid,
                'timestamp':  ts
            })

        add_event('page_view', t)
        if np.random.random() < 0.70:
            t   = t + timedelta(seconds=np.random.randint(10, 120))
            pid = np.random.choice(product_ids)
            add_event('product_view', t, pid)
            if np.random.random() < 0.35:
                t = t + timedelta(seconds=np.random.randint(5, 60))
                add_event('add_to_cart', t, pid)
                if np.random.random() < 0.55:
                    t = t + timedelta(seconds=np.random.randint(10, 90))
                    add_event('checkout_start', t, pid)
                    if np.random.random() < purchase_rate:
                        t = t + timedelta(seconds=np.random.randint(30, 300))
                        add_event('purchase', t, pid)

df_events = pd.DataFrame(events)
df_events['timestamp'] = pd.to_datetime(df_events['timestamp'])

print(f'Simulated events:  {len(df_events):,}')
print(f'Unique sessions:   {df_events["session_id"].nunique():,}')
print(f'Date range:        {df_events["timestamp"].min().date()} to {df_events["timestamp"].max().date()}')
print(f'Variant A sessions:{df_events[df_events["variant"]=="A"]["session_id"].nunique():,}')
print(f'Variant B sessions:{df_events[df_events["variant"]=="B"]["session_id"].nunique():,}')
display(df_events.head(8))

In [ ]:
# Cell 10 — Funnel analysis
funnel = df_events.groupby('event_type')['session_id'].nunique().reindex(event_types)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars   = ax.barh(event_types[::-1], funnel.values[::-1], color=colors[::-1])

for bar, val in zip(bars, funnel.values[::-1]):
    pct = f'{val / funnel.iloc[0] * 100:.1f}%'
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({pct})', va='center', fontsize=9)

ax.set_title('Conversion Funnel — Sessions by Stage', fontsize=13, fontweight='bold')
ax.set_xlabel('Unique Sessions')
ax.set_xlim(0, funnel.max() * 1.25)
plt.tight_layout()
plt.savefig('../data/sample/funnel_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nStage-to-stage conversion rates:')
for i in range(len(event_types) - 1):
    if funnel.iloc[i] > 0:
        rate = funnel.iloc[i+1] / funnel.iloc[i] * 100
        print(f'  {event_types[i]:20s} -> {event_types[i+1]:20s}: {rate:.1f}%')

In [ ]:
# Cell 11 — Save sample data
df_products.to_csv('../data/sample/products_sample.csv', index=False)
df_users[['id', 'username', 'email']].to_csv('../data/sample/users_sample.csv', index=False)
df_orders_agg.to_csv('../data/sample/orders_sample.csv', index=False)
df_events.sample(n=min(500, len(df_events)), random_state=42).to_csv(
    '../data/sample/events_sample.csv', index=False)

print('Saved to data/sample/:')
for f in sorted(os.listdir('../data/sample')):
    if f.endswith('.csv'):
        size = os.path.getsize(f'../data/sample/{f}')
        print(f'  {f:45s} {size:>8,} bytes')

## EDA Summary — Design Decisions

| Finding | Pipeline Decision |
|---------|------------------|
| No duplicate IDs in products, users, carts | Bronze append-only; Silver MERGE deduplicates on ID |
| Referential integrity intact | dbt staging `relationships` test in schema.yml |
| Carts contain nested `products` array | Silver explodes nested arrays into flat cart_items table |
| Price range $1–$1,000, no negatives | pandera contract: `price > 0` enforced at ingestion |
| No clickstream events in API | Synthetic event simulation for A/B test and funnel |
| 4 product categories | `accepted_values` dbt test on category column |

**Key business insight:** Electronics drives highest revenue despite fewer units — avg unit price is 4x higher than clothing. Highest-value intervention: electronics cart abandonment recovery.